In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, make_scorer, recall_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


c:\Users\vivek\anaconda4\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv("Telco-Customer-Churn.csv")
df = df.drop("customerID",axis=1)
# Handle TotalCharges whitespace -> NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Convert SeniorCitizen to categorical
df['SeniorCitizen'] = df['SeniorCitizen'].map({1: 'Yes', 0: 'No'})


In [3]:
X = df.drop('Churn', axis=1)
y = df['Churn'].map({"Yes":1,"No":0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [4]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_cols = [col for col in X_train.columns if col not in numeric_cols]

binary_cols = [col for col in categorical_cols if X_train[col].nunique() == 2]
multi_cols  = [col for col in categorical_cols if X_train[col].nunique() > 2]


In [5]:
# Numeric preprocessing: median impute + scaling
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Binary categorical: most frequent impute + label encode as one-hot
binary_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='if_binary', dtype=int))
])

# Multi-class categorical: most frequent impute + one-hot
multi_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# Combine everything
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('bin', binary_transformer, binary_cols),
    ('multi', multi_transformer, multi_cols)
])


In [6]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=300),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "XGBoost": xgb.XGBClassifier(eval_metric='logloss')
}

pipelines = {}
for name, model in models.items():
    pipelines[name] = ImbPipeline([
        ('preprocess', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', model)
    ])


In [7]:
# Use cross_val_score directly. ImbPipeline handles SMOTE correctly (only fitting it on train folds).
# We score on Recall (pos_label=1) to focus on catching churners.

cv_results = {}

for name, pipeline in pipelines.items():
    # scoring='recall' uses the default positive label (1), which matches our mapping.
    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='recall')
    cv_results[name] = np.mean(scores)
    print(f"{name} CV Recall: {cv_results[name]:.4f}")

Logistic Regression CV Recall: 0.8007
Decision Tree CV Recall: 0.5438
Random Forest CV Recall: 0.5833
XGBoost CV Recall: 0.5612


In [8]:
from sklearn.metrics import make_scorer, recall_score

# Make scorer with zero_division=0 to prevent nan
recall_scorer = make_scorer(recall_score, pos_label=1, zero_division=0)

# Hyperparameter grid for Logistic Regression
param_grid_lr = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],  # regularization strength
    'classifier__penalty': ['l2'],             # l2 regularization
    'classifier__solver': ['lbfgs', 'saga']    # solvers supporting l2
}

# GridSearchCV with StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    pipelines['Logistic Regression'],  # use the LR pipeline
    param_grid=param_grid_lr,
    scoring=recall_scorer,            # use recall
    cv=skf,
    n_jobs=-1
)

# Fit grid search on training set
grid_search.fit(X_train, y_train)

# Best hyperparameters and CV recall
print("Best LR params:", grid_search.best_params_)
print("Best CV Recall:", grid_search.best_score_)


Best LR params: {'classifier__C': 1, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}
Best CV Recall: 0.7906354515050168


In [9]:
# Use best estimator from grid search
final_model = grid_search.best_estimator_

# Predict on untouched test set
y_pred = final_model.predict(X_test)

# Evaluation
print("\n=== FINAL TEST SET RESULTS ===")
print(classification_report(y_test, y_pred))



=== FINAL TEST SET RESULTS ===
              precision    recall  f1-score   support

           0       0.91      0.72      0.80      1035
           1       0.51      0.80      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.74      0.75      1409

